In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.delhi_air.ml_features_6h AS
WITH b AS (
    SELECT sensor_id, location_id, location_name, provider, hour_ist, pm25_avg,
           temperature_c, humidity_pct, wind_speed_kmh, wind_direction_deg,
           pressure_hpa, precipitation_mm, boundary_layer_m
    FROM workspace.delhi_air.gold_station_hourly
)
SELECT b.sensor_id, b.location_id, b.location_name, b.provider, b.hour_ist,
       hour(b.hour_ist) AS hour_of_day,
       b.pm25_avg AS pm25_now,
       l1.pm25_avg AS pm25_lag1,
       l3.pm25_avg AS pm25_lag3,
       l6.pm25_avg AS pm25_lag6,
       l24.pm25_avg AS pm25_lag24,
       b.temperature_c, b.humidity_pct, b.wind_speed_kmh, b.wind_direction_deg,
       b.pressure_hpa, b.precipitation_mm, b.boundary_layer_m,
       f6.pm25_avg AS pm25_target_6h
FROM b
JOIN b f6  ON f6.sensor_id  = b.sensor_id AND f6.hour_ist  = b.hour_ist + INTERVAL 6 HOURS
LEFT JOIN b l1  ON l1.sensor_id  = b.sensor_id AND l1.hour_ist  = b.hour_ist - INTERVAL 1 HOURS
LEFT JOIN b l3  ON l3.sensor_id  = b.sensor_id AND l3.hour_ist  = b.hour_ist - INTERVAL 3 HOURS
LEFT JOIN b l6  ON l6.sensor_id  = b.sensor_id AND l6.hour_ist  = b.hour_ist - INTERVAL 6 HOURS
LEFT JOIN b l24 ON l24.sensor_id = b.sensor_id AND l24.hour_ist = b.hour_ist - INTERVAL 24 HOURS
WHERE b.pm25_avg IS NOT NULL AND b.temperature_c IS NOT NULL
""")

spark.sql("""
    SELECT count(*) AS rows, count(pm25_lag24) AS rows_with_lag24,
           min(hour_ist) AS first_hour, max(hour_ist) AS last_hour,
           round(avg(abs(pm25_now - pm25_target_6h)), 1) AS persistence_mae,
           round(avg(pm25_target_6h), 1) AS mean_target
    FROM workspace.delhi_air.ml_features_6h
""").show(truncate=False)